<a href="https://colab.research.google.com/github/milleau98/2026-gig-data-analysis/blob/main/notebooks/dataset-generators/google_trends.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pytrends pandas


In [3]:
import os
from getpass import getpass

TOKEN = getpass('Enter GitHub token: ')
REPO = "milleau98/2026-gig-data-analysis"

if not os.path.exists("2026-gig-data-analysis"):
    !git clone https://{TOKEN}@github.com/{REPO}.git
else:
    %cd 2026-gig-data-analysis
    !git pull

Enter GitHub token: ··········
Cloning into '2026-gig-data-analysis'...
remote: Enumerating objects: 599, done.
remote: Counting objects: 100% (158/158), done.
remote: Compressing objects: 100% (120/120), done.
remote: Total 599 (delta 98), reused 47 (delta 38), pack-reused 441 (from 1)
Receiving objects: 100% (599/599), 35.32 MiB | 15.55 MiB/s, done.
Resolving deltas: 100% (289/289), done.


In [4]:
from pytrends.request import TrendReq
import pandas as pd
import time

# connect to Google trends
pytrends = TrendReq(hl='en-US', tz=360)

anchor = 'Uber'

keyword_groups = [
    [anchor,'Lyft','DoorDash','Instacart'],
    [anchor, 'Fiverr','Upwork','side hustle','gig'],
    [anchor, 'Herbalife','Primerica','Nu skin', 'USANA'],
    [anchor, 'Etsy', 'Shopify', 'Udemy']
]

all_data = []

for i, group in enumerate(keyword_groups):
  # timeframe and geo
  pytrends.build_payload(group, timeframe='2010-01-01 2025-12-31', geo='US')

  df=pytrends.interest_over_time()

  df=df.drop(columns=['isPartial'])

  # store anchor values
  if i == 0:
    anchor_series = df[anchor]
    all_data.append(df)
  else:
    epsilon = 0.01
    scale_factor = anchor_series / (df[anchor] + epsilon)
    #scale_factor = anchor_series.div(df[anchor]).replace([float("inf"), -float("inf")], 0)
    df = df.multiply(scale_factor, axis=0)
    all_data.append(df.drop(columns=[anchor]))

# combine groups of data
final_data = pd.concat(all_data, axis=1)

final_data.head()

,Uber,Lyft,DoorDash,Instacart,Fiverr,Upwork,side hustle,gig,Herbalife,Primerica,Nu skin,USANA,Etsy,Shopify,Udemy
date,,,,,,,,,,,,,,,
2010-01-01,2,0,0,0,0.0,0.0,0.0,7.0,3.0,2.0,1.0,1.0,13.0,0.0,0.0
2010-02-01,2,0,0,0,0.0,0.0,0.0,7.0,3.0,2.0,1.0,1.0,14.0,0.0,0.0
2010-03-01,2,0,0,0,0.0,0.0,0.0,7.0,3.0,3.0,1.0,1.0,14.0,0.0,0.0
2010-04-01,2,0,0,0,0.0,0.0,0.0,7.0,3.0,3.0,0.0,1.0,13.0,0.0,0.0
2010-05-01,2,0,0,0,0.0,0.0,0.0,8.0,3.0,2.0,0.0,1.0,14.0,0.0,0.0


In [8]:
# convert weekly trends to monthly averages
monthly = final_data.resample("M").mean()

monthly['date'] = monthly.index
monthly['year'] = monthly.index.year
monthly['month'] = monthly.index.month
monthly = monthly.fillna(0)
monthly = monthly.reset_index(drop=True)

monthly.head()

/tmp/ipykernel_3367/88298119.py:2: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly = final_data.resample("M").mean()


,Uber,Lyft,DoorDash,Instacart,Fiverr,Upwork,side hustle,gig,Herbalife,Primerica,Nu skin,USANA,Etsy,Shopify,Udemy,date,year,month
0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,3.0,2.0,1.0,1.0,13.0,0.0,0.0,2010-01-31,2010,1
1,2.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,3.0,2.0,1.0,1.0,14.0,0.0,0.0,2010-02-28,2010,2
2,2.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,3.0,3.0,1.0,1.0,14.0,0.0,0.0,2010-03-31,2010,3
3,2.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,3.0,3.0,0.0,1.0,13.0,0.0,0.0,2010-04-30,2010,4
4,2.0,0.0,0.0,0.0,0.0,0.0,0.0,8.0,3.0,2.0,0.0,1.0,14.0,0.0,0.0,2010-05-31,2010,5


In [6]:
# convert weekly trends into quarterly averages
quarterly = final_data.resample("Q").mean()
quarterly.index = quarterly.index.to_period("Q").to_timestamp()
quarterly['year'] = quarterly.index.year
quarterly['quarter'] = quarterly.index.quarter
quarterly['date'] = quarterly.index
quarterly = quarterly.reset_index(drop=True)
quarterly.head()

/tmp/ipykernel_3367/3960241845.py:2: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  quarterly = final_data.resample("Q").mean()


,Uber,Lyft,DoorDash,Instacart,Fiverr,Upwork,side hustle,gig,Herbalife,Primerica,Nu skin,USANA,Etsy,Shopify,Udemy,year,quarter,date
0,2.000000,0.0,0.0,0.0,0.0,0.0,0.0,7.000000,3.000000,2.333333,1.000000,1.000000,13.666667,0.0,0.0,2010,1,2010-01-01
1,2.000000,0.0,0.0,0.0,0.0,0.0,0.0,7.666667,3.000000,2.333333,0.000000,1.000000,13.666667,0.0,0.0,2010,2,2010-04-01
2,2.333333,0.0,0.0,0.0,0.0,0.0,0.0,7.333333,3.000000,2.000000,0.000000,1.333333,15.333333,0.0,0.0,2010,3,2010-07-01
3,2.000000,0.0,0.0,0.0,0.0,0.0,0.0,8.000000,2.666667,2.000000,0.333333,0.666667,20.000000,0.0,0.0,2010,4,2010-10-01
4,2.333333,0.0,0.0,0.0,0.0,0.0,0.0,7.333333,3.000000,2.000000,0.000000,1.000000,24.666667,0.0,0.0,2011,1,2011-01-01


In [7]:
# convert weekly trends into annual averages
annual = final_data.resample("Y").mean()
annual.index = annual.index.to_period("Y").to_timestamp()
annual["year"] = annual.index.year
annual["date"] = annual.index
annual = annual.reset_index(drop=True)

annual.head()

/tmp/ipykernel_3367/646657277.py:2: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  annual = final_data.resample("Y").mean()


,Uber,Lyft,DoorDash,Instacart,Fiverr,Upwork,side hustle,gig,Herbalife,Primerica,Nu skin,USANA,Etsy,Shopify,Udemy,year,date
0,2.083333,0.000000,0.0,0.000000,0.000000,0.0,0.0,7.500000,2.916667,2.166667,0.333333,1.000000,15.666667,0.000000,0.000000,2010,2010-01-01
1,2.083333,0.000000,0.0,0.000000,0.250000,0.0,0.0,7.416667,3.500000,2.166667,0.000000,1.000000,30.166667,0.166667,0.000000,2011,2011-01-01
2,2.833333,0.000000,0.0,0.000000,1.083333,0.0,0.0,6.750000,4.833333,1.916667,0.916667,1.000000,44.666667,1.000000,0.000000,2012,2012-01-01
3,6.500000,1.333333,0.0,0.000000,1.000000,0.0,0.0,6.583333,7.000000,1.916667,1.000000,1.000000,54.000000,1.016667,0.000000,2013,2013-01-01
4,24.750000,4.416667,0.0,0.666667,1.000000,0.0,0.0,6.416667,7.583333,1.916667,0.666667,0.833333,54.878113,2.009762,1.004881,2014,2014-01-01


In [10]:
import os
os.makedirs("data/google_trends", exist_ok=True)

# create dataset files
monthly.to_csv('2026-gig-data-analysis/data/google_trends/google_trends_monthly.csv', index=False)
#quarterly.to_csv('2026-gig-data-analysis/data/google_trends/google_trends_quarterly.csv', index=False)
#annual.to_csv("2026-gig-data-analysis/data/google_trends/google_trends_annual.csv", index=False)

In [11]:
# download as csv
from google.colab import files
files.download('2026-gig-data-analysis/data/google_trends/google_trends_monthly.csv')
#files.download('2026-gig-data-analysis/data/google_trends/google_trends_quarterly.csv')
#files.download('2026-gig-data-analysis/data/google_trends/google_trends_annual.csv')




<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>